# GeoLife CP2 — Home / Office / POI Baseline

**Mục tiêu:** xây một baseline Home / Office có thể giải thích được, bắt đầu từ frozen CP1 stay events chứ không từ raw GPS.

### Câu hỏi lớn

1. user có đủ repeated history để suy semantic location không;
2. DBSCAN và complete-link trade-off ra sao khi gom recurring location;
3. UTC timestamps phải chuyển sang behavioral local time theo policy nào;
4. **timezone assignment và Beijing geographic scope có đang bị trộn thành một rule hay không;**
5. night/daytime evidence nên tính trên whole stay hay exact interval overlap;
6. khi nào nên **abstain** thay vì ép HOME/OFFICE;
7. production implementation có reproduce đúng notebook decision trên full release không.

### Nguyên tắc

```text
CP1 cleaning/stays
      ↓
user-level history
      ↓
coordinate → IANA timezone per stay
      ↓
optional geography scope (separate concern)
      ↓
compact recurring locations
      ↓
behavioral-time evidence
      ↓
abstention + evidence strength
```

> GeoLife không có direct HOME/OFFICE ground truth. Vì vậy notebook đánh giá **coverage, stability, support và plausibility**, không báo accuracy.

> HOME/OFFICE là sensitive derived locations. Không commit precise user-level inferred coordinates hoặc private caches vào repo.

### 2026-09-24 timezone review

Bản notebook này **không còn dùng `distance tới một Beijing reference point <= 100 km` như proxy cho timezone**.

Thay vào đó:

```text
mỗi stay (lat, lon)
→ timezone polygon lookup
→ IANA tzid
→ local time theo chính tzid đó
```

Sau đó mới tính user có **Asia/Shanghai-focused** hay không bằng stay-share + dwell-share. Đây là timezone concentration, **không phải Beijing administrative membership**.

Nếu project cần Beijing-only geographic cohort, geography phải là một gate riêng (ưu tiên administrative polygon), không được ngầm coi timezone polygon là Beijing boundary.

Design contract: `docs/design/03_home_office_baseline_contract.md`.


In [ ]:
from pathlib import Path
from time import perf_counter
from zoneinfo import ZoneInfo
from IPython.display import display
import os
import pickle
import subprocess
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.cluster import AgglomerativeClustering, DBSCAN

try:
    from timezonefinder import TimezoneFinder
except ImportError:
    # Pin the notebook dependency so the coordinate->timezone audit is reproducible.
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "timezonefinder==9.0.0"],
        check=True,
    )
    from timezonefinder import TimezoneFinder

REPO_URL = "https://github.com/tanh1c/geolife.git"
REPO_BRANCH = os.environ.get(
    "GEOLIFE_REPO_BRANCH",
    "main",
)
REPO_DIR = Path(os.environ.get("GEOLIFE_REPO_DIR", "/tmp/geolife"))
VOLUME_ROOT = Path("/mnt/geolife-data")
CACHE_DIR = VOLUME_ROOT / "cache" / "cp2_home_office"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

def resolve_data_root():
    env_root = os.environ.get("GEOLIFE_DATA_ROOT")
    candidates = ([Path(env_root)] if env_root else []) + [
        VOLUME_ROOT / "extracted" / "Geolife Trajectories 1.3" / "Data",
        VOLUME_ROOT / "Data",
    ]
    for candidate in candidates:
        if candidate.is_dir() and any(candidate.glob("*/Trajectory/*.plt")):
            return candidate
    for candidate in sorted(VOLUME_ROOT.glob("**/Data")):
        if candidate.is_dir() and any(candidate.glob("*/Trajectory/*.plt")):
            return candidate
    raise FileNotFoundError("GeoLife Data folder not found")

def ensure_repo():
    if (REPO_DIR / ".git").exists():
        subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin"], check=True)
        subprocess.run(["git", "-C", str(REPO_DIR), "checkout", REPO_BRANCH], check=True)
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", REPO_BRANCH], check=True)
    else:
        subprocess.run(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR)], check=True)

DATA_ROOT = resolve_data_root()
ensure_repo()
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))

from notebooks.eda_core import read_plt
from geolife.geo.distance import haversine_m
from geolife.staypoints import clean_trajectory, detect_staypoints
from geolife.model import HomeOfficeConfig, build_semantic_locations, infer_home_office

BASELINE = {
    "same_second_radius_m": 10.0,
    "max_gap_s": 300.0,
    "hard_speed_guard_kmh": 1200.0,
    "distance_threshold_m": 200.0,
    "min_dwell_s": 1200.0,
}

files = sorted(DATA_ROOT.glob("*/Trajectory/*.plt"))
print("Repo branch:", REPO_BRANCH)
print("Data root:", DATA_ROOT)
print("Trajectory files:", f"{len(files):,}")
print("Cache dir:", CACHE_DIR)
print("Frozen CP1 baseline:", BASELINE)


## 1. Materialize frozen CP1 stays ở cấp user

CP1 full-release audit đã xác nhận **5,821 stays** trên 18,670 files, nhưng cache cũ chủ yếu là per-file summary. CP2 cần actual stay rows để gom history theo user.

### Vì sao phải materialize lại actual stays?

Home/Office cần các field mà summary per-file không đủ:

- `user_id`;
- arrival / departure UTC;
- duration;
- stay representative coordinate;
- source-file lineage.

### Reproducibility gate

Section này phải kết thúc bằng:

```text
len(stays) == 5,821
```

Nếu không reconcile đúng CP1 total thì phải dừng — semantic stage không được âm thầm chạy trên một upstream dataset khác.

### Cache design

Run đầu có thể chậm vì phải đọc 18,670 `.plt` files. Notebook checkpoint mỗi 500 files và lưu final private cache bằng pandas pickle để không phụ thuộc `pyarrow`.

Cache chỉ là execution artifact trên mounted volume, không phải dataset để commit.

In [ ]:
STAYS_CACHE = CACHE_DIR / "stays_baseline_v1.pkl"
PARTIAL_CACHE = CACHE_DIR / "stays_baseline_v1.partial.pkl"
EXPECTED_CP1_STAYS = 5821

def user_id_from_path(path):
    return path.parent.parent.name

def process_file(path):
    raw = read_plt(path)[["timestamp", "latitude", "longitude"]]
    cleaned = clean_trajectory(
        raw,
        same_second_radius_m=BASELINE["same_second_radius_m"],
        max_gap_s=BASELINE["max_gap_s"],
        hard_speed_guard_kmh=BASELINE["hard_speed_guard_kmh"],
    )
    stays = detect_staypoints(
        cleaned,
        distance_threshold_m=BASELINE["distance_threshold_m"],
        min_dwell_s=BASELINE["min_dwell_s"],
    )
    if stays.empty:
        return []
    user_id = user_id_from_path(path)
    out = []
    for row in stays.itertuples(index=False):
        out.append({
            "user_id": user_id,
            "source_file": str(path),
            "sequence_id": int(row.sequence_id),
            "arrival_time_utc": row.arrival_time,
            "departure_time_utc": row.departure_time,
            "duration_s": float(row.duration_s),
            "latitude": float(row.latitude),
            "longitude": float(row.longitude),
            "n_points": int(row.n_points),
        })
    return out

if STAYS_CACHE.exists():
    stays = pd.read_pickle(STAYS_CACHE)
    print("Loaded:", STAYS_CACHE)
else:
    if PARTIAL_CACHE.exists():
        with PARTIAL_CACHE.open("rb") as f:
            partial = pickle.load(f)
        processed = set(partial["processed_files"])
        rows = list(partial["rows"])
        print("Resuming partial:", f"{len(processed):,}/{len(files):,} files")
    else:
        processed = set()
        rows = []

    t0 = perf_counter()
    completed_this_run = 0

    for path in files:
        key = str(path)
        if key in processed:
            continue

        rows.extend(process_file(path))
        processed.add(key)
        completed_this_run += 1

        if completed_this_run % 500 == 0:
            elapsed_min = (perf_counter() - t0) / 60
            overall_done = len(processed)
            rate = completed_this_run / max(elapsed_min, 1e-9)
            remaining = len(files) - overall_done
            eta_min = remaining / max(rate, 1e-9)
            print(
                f"{overall_done:,}/{len(files):,} files | "
                f"{len(rows):,} stays | "
                f"elapsed {elapsed_min:.1f} min | ETA ~{eta_min:.1f} min"
            )
            with PARTIAL_CACHE.open("wb") as f:
                pickle.dump(
                    {"processed_files": sorted(processed), "rows": rows},
                    f,
                    protocol=pickle.HIGHEST_PROTOCOL,
                )

    stays = pd.DataFrame(rows)
    stays["arrival_time_utc"] = pd.to_datetime(stays["arrival_time_utc"], utc=True)
    stays["departure_time_utc"] = pd.to_datetime(stays["departure_time_utc"], utc=True)
    stays = stays.sort_values(
        ["user_id", "arrival_time_utc", "source_file"], kind="stable"
    ).reset_index(drop=True)
    stays.to_pickle(STAYS_CACHE)
    if PARTIAL_CACHE.exists():
        PARTIAL_CACHE.unlink()
    print("Saved:", STAYS_CACHE)

print("Materialized stays:", f"{len(stays):,}")
print("Users with stays:", stays["user_id"].nunique())
assert len(stays) == EXPECTED_CP1_STAYS, (
    f"Expected {EXPECTED_CP1_STAYS} CP1 stays, got {len(stays)}"
)
display(stays.head())

### Kết luận phần 1

Full run đã reproduce chính xác:

- **5,821 stays**;
- **136 users** có ít nhất một stay.

Điều này đóng contract giữa CP1 và CP2: mọi semantic analysis về sau bắt đầu từ đúng production stay behavior đã validate.

**Không được suy ra:** 136 users có stays không có nghĩa 136 users đủ evidence để infer HOME/OFFICE. History sufficiency là gate riêng ở phần tiếp theo.

## 2. Kiểm tra mỗi user có đủ lịch sử để suy ra Home/Office hay chưa

### Vì sao cần bước này?

Home và Office là những nơi user thường **quay lại nhiều lần**.

Nếu một user chỉ có 1–2 stays, ta chưa có đủ bằng chứng để nói location nào là Home hay Office.

Ví dụ:

```text
User A:
- 1 stay ở một quán cà phê

User B:
- 20 stays lặp lại ở cùng một khu vực
```

Với User A, nếu vẫn ép model phải trả HOME/OFFICE thì rất dễ tạo ra nhãn sai.

Vì vậy trước khi làm semantic inference, ta cần kiểm tra:

> Mỗi user có bao nhiêu dữ liệu lịch sử và mức support có đủ mạnh hay không?

---

### Section này đo những gì?

Với mỗi user, ta tính:

* `stays`: tổng số stay đã phát hiện;
* `active_utc_dates`: stay xuất hiện trên bao nhiêu ngày khác nhau;
* `first_stay_utc`, `last_stay_utc`: stay đầu tiên và cuối cùng;
* `observation_span_days`: khoảng thời gian từ stay đầu đến stay cuối;
* `total_dwell_h`: tổng thời gian user ở trong các stays;
* `median_stay_min`: thời lượng stay điển hình.

Ví dụ:

```text
stays = 10
active_utc_dates = 6
observation_span_days = 30
```

nghĩa là user có 10 stays, trải trên 6 ngày khác nhau, trong khoảng 30 ngày quan sát.

---

### Vì sao cần cả `stays` và `active_utc_dates`?

Hai metric này đo hai khía cạnh khác nhau.

Ví dụ:

```text
User A:
10 stays nhưng tất cả trong cùng 1 ngày

User B:
10 stays trải trên 8 ngày
```

Cả hai đều có `10 stays`, nhưng User B cho thấy behavior lặp lại theo thời gian rõ hơn.

Vì Home/Office là hành vi lặp lại, số ngày có dữ liệu cũng quan trọng chứ không chỉ số lượng stays.

---

### Kết quả trên GeoLife

Measured support:

| history condition        |   users |
| ------------------------ | ------: |
| >=1 stay                 | **136** |
| >=2 stays                | **120** |
| >=5 stays                |  **99** |
| >=10 stays               |  **81** |
| stays trên >=2 UTC dates | **114** |
| >=5 UTC dates            |  **83** |
| >=10 UTC dates           |  **62** |

GeoLife có tổng cộng **182 users**.

Nhưng chỉ:

```text
136 / 182 users
```

có ít nhất một detected stay.

Tức là:

```text
182 - 136 = 46 users
```

không có stay nào dưới CP1 baseline.

Ngay trong 136 users có stay, mức độ dữ liệu cũng rất khác nhau:

```text
136 users có >=1 stay
120 users có >=2 stays
99 users có >=5 stays
81 users có >=10 stays
```

Nghĩa là càng yêu cầu history mạnh hơn thì số user đủ điều kiện càng giảm.

---

### Tại sao đây dẫn đến `abstention`?

Ta không muốn pipeline hoạt động theo kiểu:

```text
mọi user
→ bắt buộc phải có HOME
→ bắt buộc phải có OFFICE
```

Thay vào đó:

```text
đủ evidence
→ emit HOME/OFFICE

không đủ evidence
→ abstain
```

`abstain` nghĩa là:

> model chủ động không đưa ra nhãn vì dữ liệu hiện tại chưa đủ mạnh.

Đây là behavior mong muốn, không phải lỗi.

---

### Lưu ý về UTC dates

Ở section này, `active_utc_dates` chỉ dùng để đo **độ phủ lịch sử theo ngày**.

Ta chưa dùng nó để suy ra:

```text
ban đêm
giờ làm việc
weekday / weekend
```

vì các khái niệm đó phụ thuộc vào **local timezone**.

Timezone behavioral chỉ được áp dụng sau geography/timezone gate ở các section tiếp theo.


In [ ]:
user_history = (
    stays.assign(
        arrival_utc_date=stays["arrival_time_utc"].dt.date,
    )
    .groupby("user_id")
    .agg(
        stays=("user_id", "size"),
        active_utc_dates=("arrival_utc_date", "nunique"),
        first_stay_utc=("arrival_time_utc", "min"),
        last_stay_utc=("departure_time_utc", "max"),
        total_dwell_h=("duration_s", lambda s: s.sum() / 3600.0),
        median_stay_min=("duration_s", lambda s: s.median() / 60.0),
    )
)

user_history["observation_span_days"] = (
    user_history["last_stay_utc"] - user_history["first_stay_utc"]
).dt.total_seconds() / 86400.0

display(
    user_history[
        ["stays", "active_utc_dates", "observation_span_days", "total_dwell_h", "median_stay_min"]
    ].describe(percentiles=[.1,.25,.5,.75,.9,.95,.99])
)

print("Users with >=1 stay:", len(user_history))
for n in [2, 3, 5, 10]:
    print(f"Users with >= {n} stays:", int((user_history["stays"] >= n).sum()))
for n in [2, 3, 5, 10]:
    print(
        f"Users with stays on >= {n} distinct UTC dates:",
        int((user_history["active_utc_dates"] >= n).sum()),
    )

### Kết luận phần 2

Measured support:

| history condition | users |
|---|---:|
| >=1 stay | **136** |
| >=2 stays | **120** |
| >=5 stays | **99** |
| >=10 stays | **81** |
| stays trên >=2 UTC dates | **114** |
| >=5 UTC dates | **83** |
| >=10 UTC dates | **62** |

Release có 182 users, tức **46 users không có detected stay** dưới frozen CP1 baseline. Ngay trong 136 users còn lại, repeated-history support cũng không đồng đều.

Section này cho thấy dữ liệu GeoLife không đồng đều giữa các user.

Một số user có lịch sử rất dày, nhưng một số khác chỉ có vài stays hoặc không có stay nào.

Vì vậy CP2 phải hỗ trợ:

```text
đủ dữ liệu → infer
thiếu dữ liệu → abstain
```

Quan trọng:

> Coverage cao không đồng nghĩa với semantic certainty cao. Không nên ép HOME/OFFICE cho user chỉ để tăng coverage.

**Decision:** CP2 phải có abstention; coverage của pipeline không được đánh đồng với semantic certainty.

## 3. Thử gom các stays lặp lại thành location bằng DBSCAN

Một stay chỉ nói rằng:

> user đã dừng ở một vị trí trong một khoảng thời gian.

Nhưng Home/Office không nên được suy ra từ **một stay đơn lẻ**.  
Ta cần tìm những nơi user quay lại nhiều lần.

Ví dụ:

```text
stay 1  → gần cùng một khu vực
stay 2  → gần cùng một khu vực
stay 3  → gần cùng một khu vực
```

Ba stays này có thể được gom thành một **recurring location**.

---

### Vì sao thử DBSCAN?

Prototype đầu tiên dùng:

```text
DBSCAN
eps = 200 m
min_samples = 1
```

vì:

* không cần biết trước mỗi user có bao nhiêu locations;
* có thể cluster trực tiếp bằng Haversine distance trên latitude/longitude;
* `200 m` là spatial scale đã dùng ở CP1 nên dễ bắt đầu thử nghiệm.

Clustering được làm **riêng cho từng user**, vì location của user A và user B không liên quan trực tiếp với nhau.

---

### `eps = 200 m` thực sự có nghĩa gì?

Điểm rất dễ hiểu nhầm là:

```text
eps = 200 m
```

**không có nghĩa toàn bộ cluster phải nằm gọn trong đường kính 200 m.**

DBSCAN chỉ yêu cầu các points có thể nối với nhau qua các bước gần nhau.

Ví dụ:

```text
A --180m-- B --180m-- C --180m-- D
```

Mỗi cặp hàng xóm đều cách nhau dưới 200 m:

```text
A ↔ B < 200 m
B ↔ C < 200 m
C ↔ D < 200 m
```

nên DBSCAN có thể đưa tất cả vào cùng một cluster.

Nhưng:

```text
A ↔ D
```

có thể xa hơn 200 m rất nhiều.

Hiện tượng này gọi là **chaining**.

---

In [ ]:
EARTH_RADIUS_M = 6_371_008.8
LOCATION_EPS_M = 200.0

def cluster_user_stays(group, eps_m=LOCATION_EPS_M):
    g = group.sort_values("arrival_time_utc", kind="stable").copy()
    coords_rad = np.radians(g[["latitude", "longitude"]].to_numpy(dtype=float))
    labels = DBSCAN(
        eps=eps_m / EARTH_RADIUS_M,
        min_samples=1,
        metric="haversine",
        algorithm="ball_tree",
    ).fit_predict(coords_rad)
    g["location_id"] = labels.astype(int)
    return g

cluster_parts = []
for user_id, group in stays.groupby("user_id", sort=True):
    clustered_user = cluster_user_stays(group)
    cluster_parts.append(clustered_user)

clustered = pd.concat(cluster_parts, ignore_index=True) if cluster_parts else stays.copy()

location_rows = []
for (user_id, location_id), g in clustered.groupby(["user_id", "location_id"], sort=True):
    lat = float(g["latitude"].median())
    lon = float(g["longitude"].median())
    radii = np.asarray(
        haversine_m(
            g["latitude"].to_numpy(dtype=float),
            g["longitude"].to_numpy(dtype=float),
            lat,
            lon,
        ),
        dtype=float,
    )
    location_rows.append({
        "user_id": user_id,
        "location_id": int(location_id),
        "latitude": lat,
        "longitude": lon,
        "stay_count": len(g),
        "active_utc_dates": g["arrival_time_utc"].dt.date.nunique(),
        "total_dwell_h": g["duration_s"].sum() / 3600.0,
        "max_radius_m": float(np.max(radii)) if len(radii) else 0.0,
    })

locations = pd.DataFrame(location_rows)

print("Users:", locations["user_id"].nunique())
print("Candidate locations:", len(locations))
print("Recurring locations (>=2 stays):", int((locations["stay_count"] >= 2).sum()))
print("Users with >=1 recurring location:", locations.loc[
    locations["stay_count"] >= 2, "user_id"
].nunique())

display(
    locations[
        ["stay_count", "active_utc_dates", "total_dwell_h", "max_radius_m"]
    ].describe(percentiles=[.5,.75,.9,.95,.99])
)

display(
    locations.sort_values("max_radius_m", ascending=False).head(20)
)

### Kết quả prototype `eps=200 m`

Trên toàn bộ 5,821 stays, prototype DBSCAN ban đầu cho thấy GeoLife có repeated-location structure khá rõ.

Điểm quan trọng của prototype không phải con số cluster cuối cùng, mà là một semantics dễ hiểu nhầm:

```text
DBSCAN eps = 200 m
≠
cluster diameter <= 200 m
```

DBSCAN kiểm soát **local connectivity**, nên chaining có thể làm cluster rộng hơn nhiều so với `eps`.

Vì vậy sau timezone audit, notebook sẽ chạy DBSCAN sensitivity và complete-link trên **cùng candidate semantic cohort mới**.

> Các con số recurring-user / cluster-diameter cũ từ Beijing-radius cohort không được reuse làm kết luận cho cohort mới. Phải rerun các cell phía dưới.


### Tóm tắt trước khi xử lý timezone và recurring location

Đến đây ta biết:

```text
5,821 CP1 stays
136 users có ít nhất một stay
```

Prototype clustering cho thấy nhiều user có repeated locations, nhưng DBSCAN có thể chaining nên recurring-location contract vẫn phải audit riêng.

### Vấn đề timezone cần sửa

Phần lớn GeoLife tập trung quanh Beijing nhưng dataset có travel/outlier locations. Vì thế không thể:

```text
mọi timestamp
→ +8 giờ
→ coi như Beijing local time
```

Quan trọng hơn, **khoảng cách tới một điểm ở Beijing cũng không phải timezone boundary**.

Bước tiếp theo vì vậy tách hai vấn đề:

```text
A. timezone assignment
   stay (lat, lon) → IANA timezone polygon

B. geographic product scope
   Beijing-only hay không là một policy riêng
```

Notebook hiện tại audit A trước. Candidate user cohort được định nghĩa là **Asia/Shanghai-focused** theo tỷ lệ stay + dwell, nhưng travel stays vẫn giữ timezone thật của chính chúng.


## 4. Timezone audit — tách khỏi geography heuristic

GeoLife PLT timestamps là UTC/GMT, trong khi HOME/OFFICE là behavioral-time concepts.

### Vấn đề của rule cũ

Rule cũ dùng:

```text
distance tới Beijing reference point
<= 100 km
```

rồi mới cho phép `Asia/Shanghai`.

Cách đó hữu ích như một **engineering geography scope**, nhưng không phải timezone boundary. Một timezone không được định nghĩa bằng một vòng tròn quanh thành phố.

### Rule mới

Notebook dùng coordinate của **từng stay**:

```text
(latitude, longitude)
        ↓
timezone polygon lookup
        ↓
IANA timezone id
        ↓
ZoneInfo(tzid)
        ↓
local behavioral time
```

Implementation dùng `timezonefinder`, chạy offline sau khi package được cài trong environment.

Sau đó mới xét user-level concentration:

```text
stay_share_in_Asia/Shanghai
dwell_share_in_Asia/Shanghai
```

Đây là câu hỏi:

> user có chủ yếu hoạt động trong timezone chứa Beijing hay không?

Nó **không** trả lời:

> user có nằm trong địa giới hành chính Beijing hay không?

Nếu cần Beijing-only geography, phải audit riêng bằng geographic boundary/polygon.

Reference:
- IANA tz theory: https://www.iana.org/time-zones/theory
- timezonefinder: https://pypi.org/project/timezonefinder/


In [ ]:
spatial_summary = stays[["latitude", "longitude"]].describe(
    percentiles=[.01,.05,.25,.5,.75,.95,.99]
)
display(spatial_summary)

user_centers = (
    stays.groupby("user_id")[["latitude", "longitude"]]
    .median()
    .rename(columns={"latitude":"median_latitude","longitude":"median_longitude"})
)
display(user_centers.describe(percentiles=[.01,.05,.25,.5,.75,.95,.99]))

sample_n = min(5000, len(stays))
plot_sample = stays.sample(sample_n, random_state=42) if sample_n else stays
fig, ax = plt.subplots(figsize=(9, 6))
ax.scatter(plot_sample["longitude"], plot_sample["latitude"], s=8, alpha=0.35)
ax.set(
    title="Stay-point spatial coverage (sample; UTC semantics not yet converted)",
    xlabel="longitude",
    ylabel="latitude",
)
plt.show()

print("TIMEZONE POLICY STATUS: coordinate-to-IANA audit required below")
print("Do not run Home/Office time-of-day scoring until per-stay timezone lookup is complete.")


### Kết quả spatial context

Phần lớn stay points tập trung rất rõ quanh Beijing, nhưng dataset vẫn có stays ở nhiều nơi khác.

Điều này chỉ support kết luận:

> **không được áp một timezone duy nhất cho toàn bộ release.**

Nó không support việc lấy một điểm ở Beijing rồi giả định một radius là timezone boundary.

Vì vậy spatial scatter ở đây chỉ là context; timezone assignment thực sự được làm bằng coordinate-to-IANA lookup ở phần tiếp theo.


### 4.1 Gán IANA timezone cho từng stay từ latitude / longitude

Thay vì hỏi:

```text
stay có cách Beijing center <= 100 km không?
```

ta hỏi trực tiếp:

```text
(lat, lon) của stay này
nằm trong timezone polygon nào?
```

`timezonefinder` trả về IANA timezone id, ví dụ:

```text
Asia/Shanghai
Asia/Tokyo
America/Los_Angeles
...
```

Điểm cần nhớ:

- IANA tz database **không cung cấp một rectangle lat/lon đơn giản cho từng timezone**;
- practical coordinate lookup là point-in-polygon trên timezone boundary data;
- timezone id của stay và Beijing geographic membership là hai semantics khác nhau.

### User-level candidate

Để giữ philosophy abstention của CP2, ta vẫn dùng hai tỷ lệ:

```text
target_tz_stay_share
= tỷ lệ stays của user có tzid = Asia/Shanghai

target_tz_dwell_share
= tỷ lệ total dwell time của user có tzid = Asia/Shanghai
```

Ta audit các mức:

```text
50% / 80% / 90% / 95%
```

Candidate migration giữ `80% / 80%` để thay **một tầng logic** trước, không retune đồng thời mọi threshold.

> User được chọn ở đây là **Asia/Shanghai-focused**, không được gọi là “đã xác nhận ở Beijing”.


In [ ]:
TARGET_PRIMARY_TZ = "Asia/Shanghai"
MIN_TZ_SHARE_VALUES = [0.50, 0.80, 0.90, 0.95]

# Migration candidate: keep the old 80/80 concentration threshold
# while replacing the radius proxy with actual coordinate->timezone lookup.
CANDIDATE_MIN_TZ_STAY_SHARE = 0.80
CANDIDATE_MIN_TZ_DWELL_SHARE = 0.80

timezone_finder = TimezoneFinder()

# -------------------------------------------------------------------
# 1. Per-stay IANA timezone lookup from WGS84 coordinates
# -------------------------------------------------------------------

stays_tz = stays.copy()

stays_tz["timezone_id"] = [
    timezone_finder.timezone_at(
        lng=float(lon),
        lat=float(lat),
    )
    for lat, lon in zip(
        stays_tz["latitude"].to_numpy(dtype=float),
        stays_tz["longitude"].to_numpy(dtype=float),
    )
]

unresolved_tz = int(stays_tz["timezone_id"].isna().sum())

print("Timezone lookup:")
print("  stays:", len(stays_tz))
print("  unresolved timezone:", unresolved_tz)

timezone_summary = (
    stays_tz.assign(dwell_h=stays_tz["duration_s"] / 3600.0)
    .groupby("timezone_id", dropna=False)
    .agg(
        stays=("user_id", "size"),
        users=("user_id", "nunique"),
        dwell_h=("dwell_h", "sum"),
    )
    .sort_values(["stays", "dwell_h"], ascending=False)
)

print("\nTop timezone IDs by stay count:")
display(timezone_summary.head(20))


# -------------------------------------------------------------------
# 2. User-level timezone concentration
# -------------------------------------------------------------------

stays_tz["in_target_timezone"] = (
    stays_tz["timezone_id"] == TARGET_PRIMARY_TZ
)

stays_tz["target_timezone_dwell_s"] = np.where(
    stays_tz["in_target_timezone"],
    stays_tz["duration_s"],
    0.0,
)

user_timezone = (
    stays_tz.groupby("user_id")
    .agg(
        total_stays=("user_id", "size"),
        target_tz_stays=("in_target_timezone", "sum"),
        total_dwell_s=("duration_s", "sum"),
        target_tz_dwell_s=("target_timezone_dwell_s", "sum"),
        distinct_timezones=("timezone_id", "nunique"),
    )
)

user_timezone["target_tz_stay_share"] = (
    user_timezone["target_tz_stays"]
    / user_timezone["total_stays"]
)

user_timezone["target_tz_dwell_share"] = np.where(
    user_timezone["total_dwell_s"] > 0,
    user_timezone["target_tz_dwell_s"]
    / user_timezone["total_dwell_s"],
    0.0,
)


# -------------------------------------------------------------------
# 3. Sensitivity over concentration threshold
# -------------------------------------------------------------------

total_users = int(stays_tz["user_id"].nunique())
total_stays = int(len(stays_tz))

timezone_sensitivity_rows = []

for min_share in MIN_TZ_SHARE_VALUES:
    eligible = (
        (user_timezone["target_tz_stay_share"] >= min_share)
        & (user_timezone["target_tz_dwell_share"] >= min_share)
    )
    eligible_ids = user_timezone.index[eligible]
    eligible_stay_mask = stays_tz["user_id"].isin(eligible_ids)

    timezone_sensitivity_rows.append(
        {
            "min_both_shares": min_share,
            "eligible_users": int(eligible.sum()),
            "eligible_user_rate": float(eligible.mean()),
            "all_stays_from_eligible_users": int(eligible_stay_mask.sum()),
            "eligible_user_stay_rate": float(eligible_stay_mask.mean()),
            "target_tz_stays_from_eligible_users": int(
                (eligible_stay_mask & stays_tz["in_target_timezone"]).sum()
            ),
            "non_target_tz_stays_from_eligible_users": int(
                (eligible_stay_mask & ~stays_tz["in_target_timezone"]).sum()
            ),
        }
    )

timezone_sensitivity = pd.DataFrame(timezone_sensitivity_rows)

print("\nAsia/Shanghai concentration sensitivity:")
display(
    timezone_sensitivity.style.format(
        {
            "min_both_shares": "{:.0%}",
            "eligible_user_rate": "{:.2%}",
            "eligible_user_stay_rate": "{:.2%}",
        }
    )
)


# -------------------------------------------------------------------
# 4. Candidate cohort at 80/80
# -------------------------------------------------------------------

user_timezone["target_timezone_candidate"] = (
    (user_timezone["target_tz_stay_share"] >= CANDIDATE_MIN_TZ_STAY_SHARE)
    & (user_timezone["target_tz_dwell_share"] >= CANDIDATE_MIN_TZ_DWELL_SHARE)
)

primary_tz_user_ids = user_timezone.index[
    user_timezone["target_timezone_candidate"]
]

candidate_user_mask = stays_tz["user_id"].isin(primary_tz_user_ids)

# Important change from v1:
# keep ALL resolved-timezone stays of eligible users.
# Travel stays are no longer discarded just because they are outside a Beijing radius.
stays_semantic = stays_tz[
    candidate_user_mask
    & stays_tz["timezone_id"].notna()
].copy()

candidate_travel_stays = stays_semantic[
    stays_semantic["timezone_id"] != TARGET_PRIMARY_TZ
].copy()

print("\nCandidate timezone-focused cohort:")
print(
    "Policy:",
    f"target_tz={TARGET_PRIMARY_TZ}",
    f"+ stay_share >= {CANDIDATE_MIN_TZ_STAY_SHARE:.0%}",
    f"+ dwell_share >= {CANDIDATE_MIN_TZ_DWELL_SHARE:.0%}",
)
print("Eligible users:", len(primary_tz_user_ids), "/", total_users)
print("Resolved stays retained from eligible users:", len(stays_semantic), "/", total_stays)
print("Travel / non-target-timezone stays retained:", len(candidate_travel_stays))

print("\nStrongest target-timezone users:")
display(
    user_timezone[
        [
            "total_stays",
            "target_tz_stays",
            "target_tz_stay_share",
            "target_tz_dwell_share",
            "distinct_timezones",
            "target_timezone_candidate",
        ]
    ]
    .sort_values(
        [
            "target_timezone_candidate",
            "target_tz_stay_share",
            "target_tz_dwell_share",
        ],
        ascending=[False, False, False],
    )
    .head(30)
)


### Vì sao vẫn thử `80% / 80%`?

`80% / 80%` ở đây **không còn đi kèm radius 100 km**.

Nó chỉ có nghĩa:

```text
>=80% stays của user nằm trong Asia/Shanghai
AND
>=80% dwell time của user nằm trong Asia/Shanghai
```

Ta giữ cùng concentration threshold như v1 để migration dễ diễn giải:

```text
old:
Beijing-distance proxy + 80/80

new candidate:
actual timezone lookup + 80/80
```

Nhờ vậy, khi output thay đổi, ta biết thay đổi chủ yếu đến từ **cách xác định timezone**, không phải vì đồng thời retune cả share threshold.

Không có timezone/Home/Office ground truth để gọi `80%` là optimum. Bảng sensitivity phía trên phải được đọc như coverage trade-off.


### Kết luận timezone gate mới

Candidate mới hoạt động ở hai mức:

```text
stay-level
→ (lat, lon) → IANA timezone id

user-level
→ user có >=80% stay share
  và >=80% dwell share
  trong Asia/Shanghai?
```

Điểm khác v1 rất quan trọng:

```text
user Asia/Shanghai-focused
+ có travel stay ở Tokyo

→ user vẫn đủ điều kiện
→ Tokyo stay vẫn được giữ
→ Tokyo stay dùng Asia/Tokyo local time
```

Travel observation không còn bị loại chỉ vì nó ở ngoài một Beijing radius.

### Điều notebook này KHÔNG claim

`Asia/Shanghai-focused` **không đồng nghĩa** `Beijing resident`.

`Asia/Shanghai` cover một vùng lớn hơn Beijing. Nếu product requirement là “chỉ infer Beijing users”, phải thêm một **geography gate riêng**, ưu tiên Beijing administrative polygon.

**Decision của notebook candidate:** dùng timezone polygon cho local-time correctness; geography scope không được ngầm nhét vào timezone rule.


### 4.2 Chuyển UTC sang local time theo timezone của từng stay

Mỗi retained stay đã có `timezone_id`.

Ta convert:

```text
arrival_time_utc
departure_time_utc
        ↓
ZoneInfo(timezone_id)
        ↓
local wall-clock time
```

Vì một DataFrame có thể chứa nhiều timezone khác nhau, notebook lưu:

```text
timezone_id
arrival_time_local
departure_time_local
```

trong đó `arrival_time_local` / `departure_time_local` là **timezone-naive local wall time**, còn timezone identity được giữ riêng trong `timezone_id`.

Cách này giúp `.dt.date / .dt.hour / weekday` hoạt động ổn định trên một cột pandas duy nhất mà vẫn không làm mất timezone provenance.

> Không cộng thủ công `+8h`. Mỗi stay dùng timezone của chính coordinate đó.


In [ ]:
def utc_to_local_wall_time(timestamp_utc, timezone_id):
    if pd.isna(timezone_id):
        return pd.NaT

    ts = pd.Timestamp(timestamp_utc)
    if ts.tzinfo is None:
        ts = ts.tz_localize("UTC")

    return (
        ts.tz_convert(ZoneInfo(str(timezone_id)))
        .tz_localize(None)
    )


stays_semantic["arrival_time_local"] = pd.to_datetime(
    [
        utc_to_local_wall_time(ts, tzid)
        for ts, tzid in zip(
            stays_semantic["arrival_time_utc"],
            stays_semantic["timezone_id"],
        )
    ]
)

stays_semantic["departure_time_local"] = pd.to_datetime(
    [
        utc_to_local_wall_time(ts, tzid)
        for ts, tzid in zip(
            stays_semantic["departure_time_utc"],
            stays_semantic["timezone_id"],
        )
    ]
)

stays_semantic["arrival_local_date"] = (
    stays_semantic["arrival_time_local"].dt.date
)
stays_semantic["arrival_local_hour"] = (
    stays_semantic["arrival_time_local"].dt.hour
)
stays_semantic["arrival_local_weekday"] = (
    stays_semantic["arrival_time_local"].dt.weekday
)

assert stays_semantic["arrival_time_local"].notna().all()
assert stays_semantic["departure_time_local"].notna().all()

print("Target primary timezone:", TARGET_PRIMARY_TZ)
print("Users in candidate cohort:", stays_semantic["user_id"].nunique())
print("Stays retained for semantic analysis:", len(stays_semantic))
print(
    "Distinct IANA timezones retained:",
    stays_semantic["timezone_id"].nunique(),
)

print("\nTimezone mix inside candidate users:")
display(
    stays_semantic["timezone_id"]
    .value_counts()
    .rename("stays")
    .to_frame()
    .head(20)
)

print("\nSample local-time conversion (coordinates omitted):")
display(
    stays_semantic[
        [
            "user_id",
            "timezone_id",
            "arrival_time_utc",
            "arrival_time_local",
            "departure_time_local",
            "duration_s",
        ]
    ].head(10)
)

print("TIMEZONE POLICY STATUS: candidate v2 audit path ready.")
print("Production v1 remains unchanged until downstream sensitivity/parity is rerun.")


### 4.3 DBSCAN `eps` sensitivity trên timezone-resolved candidate cohort

Ta chạy DBSCAN sensitivity trên **cùng `stays_semantic`** vừa được:

```text
coordinate → timezone lookup
→ user Asia/Shanghai-focused gate
→ per-stay local time
```

Các mức:

```text
eps = 10 / 20 / 30 / 50 / 100 / 150 / 200 m
```

Với mỗi `eps`, đo tổng locations, recurring locations, recurring users và cluster diameter.

Các output cũ từ 4,197-radius cohort không còn được coi là hiện hành; phải dùng kết quả rerun của bảng bên dưới.


In [ ]:
DBSCAN_EPS_VALUES_M = [10.0, 20.0, 30.0, 50.0, 100.0, 150.0, 200.0]

def summarize_dbscan_eps(stays_input, eps_m):
    location_rows = []

    for user_id, group in stays_input.groupby("user_id", sort=True):
        g = group.sort_values("arrival_time_utc", kind="stable").copy()

        lat = g["latitude"].to_numpy(dtype=float)
        lon = g["longitude"].to_numpy(dtype=float)

        coords_rad = np.radians(
            g[["latitude", "longitude"]].to_numpy(dtype=float)
        )

        labels = DBSCAN(
            eps=eps_m / EARTH_RADIUS_M,
            min_samples=1,
            metric="haversine",
            algorithm="ball_tree",
        ).fit_predict(coords_rad)

        distances = np.asarray(
            haversine_m(
                lat[:, None],
                lon[:, None],
                lat[None, :],
                lon[None, :],
            ),
            dtype=float,
        )

        for location_id in np.unique(labels):
            member_idx = np.flatnonzero(labels == location_id)
            members = g.iloc[member_idx]

            diameter_m = (
                float(distances[np.ix_(member_idx, member_idx)].max())
                if len(member_idx) > 1
                else 0.0
            )

            location_rows.append({
                "user_id": user_id,
                "location_id": int(location_id),
                "stay_count": len(member_idx),
                "active_local_dates": members["arrival_local_date"].nunique(),
                "total_dwell_h": members["duration_s"].sum() / 3600.0,
                "diameter_m": diameter_m,
            })

    locations_eps = pd.DataFrame(location_rows)
    recurring = locations_eps["stay_count"] >= 2
    recurring_locations = locations_eps.loc[recurring].copy()

    summary = {
        "eps_m": eps_m,
        "locations": len(locations_eps),
        "recurring_locations": int(recurring.sum()),
        "users_with_recurring_location": int(
            recurring_locations["user_id"].nunique()
        ),
        "median_locations_per_user": float(
            locations_eps.groupby("user_id").size().median()
        ),
        "median_recurring_diameter_m": float(
            recurring_locations["diameter_m"].median()
        ) if len(recurring_locations) else np.nan,
        "p95_recurring_diameter_m": float(
            recurring_locations["diameter_m"].quantile(0.95)
        ) if len(recurring_locations) else np.nan,
        "max_recurring_diameter_m": float(
            recurring_locations["diameter_m"].max()
        ) if len(recurring_locations) else np.nan,
        "recurring_clusters_le_200_rate": float(
            (recurring_locations["diameter_m"] <= 200.0).mean()
        ) if len(recurring_locations) else np.nan,
        "recurring_clusters_gt_200": int(
            (recurring_locations["diameter_m"] > 200.0).sum()
        ) if len(recurring_locations) else 0,
    }

    return locations_eps, summary


dbscan_sensitivity_rows = []
dbscan_artifacts = {}

for eps_m in DBSCAN_EPS_VALUES_M:
    locations_eps, summary = summarize_dbscan_eps(
        stays_semantic,
        eps_m,
    )
    dbscan_artifacts[eps_m] = locations_eps
    dbscan_sensitivity_rows.append(summary)

dbscan_eps_sensitivity = pd.DataFrame(dbscan_sensitivity_rows)

display(
    dbscan_eps_sensitivity[
        [
            "eps_m",
            "locations",
            "recurring_locations",
            "users_with_recurring_location",
            "median_locations_per_user",
            "median_recurring_diameter_m",
            "p95_recurring_diameter_m",
            "max_recurring_diameter_m",
            "recurring_clusters_le_200_rate",
            "recurring_clusters_gt_200",
        ]
    ]
)


### Cách đọc DBSCAN sensitivity sau timezone migration

Không reuse các số DBSCAN cũ vì input cohort đã đổi.

Khi rerun, đọc trade-off theo hai phía:

```text
eps nhỏ
→ thường compact hơn
→ nhưng dễ fragment recurring place

eps lớn
→ thường tăng recurrence coverage
→ nhưng chaining có thể làm diameter lớn
```

Điểm cần kiểm tra là recurring-user coverage, `p95/max diameter` và số recurring clusters vượt `200m`.

DBSCAN vẫn là benchmark representation; bảng này không đo supervised accuracy.


### 4.4 Complete-link recurring-location audit

Sau DBSCAN, ta giữ complete-link làm comparator vì threshold có hard compactness semantics:

```text
complete-link threshold = 200 m
→ maximum pairwise distance trong cluster <= 200 m
```

Ta vẫn audit `100 / 200 / 300 m` nhưng phải rerun trên timezone-resolved candidate cohort mới trước khi khẳng định `200m` còn giữ cùng coverage trade-off như v1.


In [ ]:
COMPLETE_LINK_THRESHOLDS_M = [100.0, 200.0, 300.0]
CANDIDATE_COMPLETE_LINK_M = 200.0

def pairwise_haversine_matrix_m(group):
    lat = group["latitude"].to_numpy(dtype=float)
    lon = group["longitude"].to_numpy(dtype=float)
    return np.asarray(
        haversine_m(
            lat[:, None],
            lon[:, None],
            lat[None, :],
            lon[None, :],
        ),
        dtype=float,
    )

def complete_link_user(group, threshold_m):
    g = group.sort_values("arrival_time_utc", kind="stable").copy()
    n = len(g)
    if n == 1:
        g["location_id"] = 0
        return g, np.zeros((1, 1), dtype=float)

    distances = pairwise_haversine_matrix_m(g)
    labels = AgglomerativeClustering(
        n_clusters=None,
        metric="precomputed",
        linkage="complete",
        distance_threshold=threshold_m,
    ).fit_predict(distances)
    g["location_id"] = labels.astype(int)
    return g, distances

def summarize_complete_link(threshold_m):
    clustered_parts = []
    location_rows = []

    for user_id, group in stays_semantic.groupby("user_id", sort=True):
        clustered_user, distances = complete_link_user(group, threshold_m)
        clustered_parts.append(clustered_user)

        labels = clustered_user["location_id"].to_numpy(dtype=int)
        for location_id in np.unique(labels):
            member_idx = np.flatnonzero(labels == location_id)
            members = clustered_user.iloc[member_idx]
            diameter_m = (
                float(distances[np.ix_(member_idx, member_idx)].max())
                if len(member_idx) > 1
                else 0.0
            )
            location_rows.append({
                "user_id": user_id,
                "location_id": int(location_id),
                "latitude": float(members["latitude"].median()),
                "longitude": float(members["longitude"].median()),
                "stay_count": len(members),
                "active_local_dates": members["arrival_local_date"].nunique(),
                "total_dwell_h": members["duration_s"].sum() / 3600.0,
                "diameter_m": diameter_m,
            })

    clustered_all = pd.concat(clustered_parts, ignore_index=True)
    locations_all = pd.DataFrame(location_rows)

    recurring = locations_all["stay_count"] >= 2
    return clustered_all, locations_all, {
        "threshold_m": threshold_m,
        "locations": len(locations_all),
        "recurring_locations": int(recurring.sum()),
        "users_with_recurring_location": int(
            locations_all.loc[recurring, "user_id"].nunique()
        ),
        "median_locations_per_user": float(
            locations_all.groupby("user_id").size().median()
        ),
        "p95_diameter_m": float(locations_all["diameter_m"].quantile(0.95)),
        "max_diameter_m": float(locations_all["diameter_m"].max()),
    }

cluster_sensitivity_rows = []
cluster_artifacts = {}

for threshold_m in COMPLETE_LINK_THRESHOLDS_M:
    clustered_threshold, locations_threshold, summary = summarize_complete_link(
        threshold_m
    )
    cluster_sensitivity_rows.append(summary)
    cluster_artifacts[threshold_m] = (
        clustered_threshold,
        locations_threshold,
    )

complete_link_sensitivity = pd.DataFrame(cluster_sensitivity_rows)
display(complete_link_sensitivity)

semantic_stays, semantic_locations = cluster_artifacts[CANDIDATE_COMPLETE_LINK_M]

assert semantic_locations["diameter_m"].max() <= CANDIDATE_COMPLETE_LINK_M + 1e-6

print("Candidate complete-link threshold:", CANDIDATE_COMPLETE_LINK_M, "m")
print("Semantic locations:", len(semantic_locations))
print(
    "Recurring semantic locations (>=2 stays):",
    int((semantic_locations["stay_count"] >= 2).sum()),
)
print(
    "Users with recurring semantic location:",
    semantic_locations.loc[
        semantic_locations["stay_count"] >= 2, "user_id"
    ].nunique(),
)
print(
    "Max verified cluster diameter (m):",
    semantic_locations["diameter_m"].max(),
)

display(
    semantic_locations.sort_values(
        ["stay_count", "total_dwell_h"],
        ascending=False,
    ).head(30)
)


### Diễn giải complete-link sau timezone migration

Không dùng lại các con số `67 / 73 users` hay `486 recurring locations` từ cohort cũ.

Sau khi rerun, kiểm tra:

```text
100 → 200 m
coverage tăng bao nhiêu?

200 → 300 m
có thêm recurring users không,
hay chỉ merge thành location rộng hơn?
```

`200m` được giữ làm **control candidate** trong migration để tránh đổi đồng thời cả timezone rule lẫn clustering rule. Nếu sensitivity mới thay đổi đáng kể, threshold phải được review lại thay vì tự động freeze.


### 4.5 So sánh DBSCAN và complete-link trên cùng candidate cohort

So sánh hai representations trên **cùng timezone-resolved input**.

DBSCAN: `30 / 100 / 200 m`. Complete-link control: `200 m`.

Metrics gồm total locations, recurring locations/users, p95/max diameter và tỷ lệ clusters `<=200m`.

Mục tiêu là kiểm tra representation trade-off sau khi timezone cohort thay đổi. Không được dùng bảng cũ để claim phương pháp nào “tốt hơn” nếu chưa rerun.


In [ ]:
dbscan_compare = (
    dbscan_eps_sensitivity[
        dbscan_eps_sensitivity["eps_m"].isin([30.0, 100.0, 200.0])
    ][
        [
            "eps_m",
            "locations",
            "recurring_locations",
            "users_with_recurring_location",
            "p95_recurring_diameter_m",
            "max_recurring_diameter_m",
            "recurring_clusters_le_200_rate",
        ]
    ]
    .copy()
)

dbscan_compare["method"] = "DBSCAN"
dbscan_compare["setting_m"] = dbscan_compare["eps_m"]
dbscan_compare = dbscan_compare.rename(
    columns={
        "p95_recurring_diameter_m": "p95_diameter_m",
        "max_recurring_diameter_m": "max_diameter_m",
        "recurring_clusters_le_200_rate": "clusters_le_200_rate",
    }
)

complete_compare = complete_link_sensitivity[
    complete_link_sensitivity["threshold_m"] == CANDIDATE_COMPLETE_LINK_M
].copy()

complete_compare["method"] = "complete-link"
complete_compare["setting_m"] = complete_compare["threshold_m"]
complete_compare["clusters_le_200_rate"] = (
    complete_compare["max_diameter_m"] <= CANDIDATE_COMPLETE_LINK_M + 1e-6
).astype(float)

clustering_benchmark = pd.concat(
    [
        dbscan_compare[
            [
                "method",
                "setting_m",
                "locations",
                "recurring_locations",
                "users_with_recurring_location",
                "p95_diameter_m",
                "max_diameter_m",
                "clusters_le_200_rate",
            ]
        ],
        complete_compare[
            [
                "method",
                "setting_m",
                "locations",
                "recurring_locations",
                "users_with_recurring_location",
                "p95_diameter_m",
                "max_diameter_m",
                "clusters_le_200_rate",
            ]
        ],
    ],
    ignore_index=True,
)

display(clustering_benchmark)


### Kết quả benchmark — phải đọc từ bảng rerun

Sau timezone migration, các số benchmark cũ có thể thay đổi vì input users/stays đã thay đổi.

Decision logic vẫn là:

```text
coverage tương đương
+ complete-link giữ hard diameter contract
→ complete-link có lợi thế engineering

DBSCAN tăng coverage đáng kể
→ trade-off cần review lại
```

Không có Home/Office ground truth, nên benchmark này chỉ đánh giá **coverage + spatial compactness**, không đánh giá semantic accuracy.

Trong notebook candidate, `complete-link 200m` vẫn được giữ như fixed control để isolate tác động của timezone change.


## 5. Candidate CP2 timezone-v2 configuration

Production CP2 v1 hiện vẫn dùng Beijing-radius geography/timezone rule trong `src/`.

Notebook này đang audit một candidate thay thế:

```text
old v1
Beijing reference point + 100 km
→ filter stays
→ Asia/Shanghai

candidate v2
(lat, lon) per stay
→ IANA timezone polygon lookup
→ local time per stay
→ Asia/Shanghai-focused user concentration
```

Để isolate tác động, các policy phía sau tạm giữ như v1: complete-link `200m` và cùng HOME/OFFICE windows + emission gates.

Các giá trị này là **fixed controls trong migration audit**, chưa được refreeze cho candidate mới cho tới khi Run All xong.


In [ ]:
CP2_TIMEZONE_V2_CANDIDATE = {
    "timezone_assignment": "coordinate_polygon_lookup",
    "timezone_lookup_library": "timezonefinder==9.0.0",
    "target_primary_timezone": TARGET_PRIMARY_TZ,
    "target_timezone_min_stay_share": CANDIDATE_MIN_TZ_STAY_SHARE,
    "target_timezone_min_dwell_share": CANDIDATE_MIN_TZ_DWELL_SHARE,
    "retain_resolved_travel_stays": True,
    "beijing_geography_gate": "not_applied_in_this_candidate",
    "clustering_method_control": "complete_link",
    "candidate_location_complete_link_m": CANDIDATE_COMPLETE_LINK_M,
    "home_night_start_hour": 21,
    "home_night_end_hour": 6,
    "office_start_hour": 9,
    "office_end_hour": 17,
    "office_weekdays": [0, 1, 2, 3, 4],
    "home_min_dates": 3,
    "home_min_share": 0.50,
    "home_min_margin": 0.20,
    "office_min_dates": 3,
    "office_min_share": 0.30,
    "office_min_margin": 0.10,
    "support_saturation_dates": 5,
}

display(pd.Series(CP2_TIMEZONE_V2_CANDIDATE, name="candidate_value"))
print("STATUS: timezone-v2 candidate only; production v1 remains frozen until rerun + parity review.")


## 6. Home / Office scoring audit

### Vì sao không chỉ nhìn `arrival hour`?

Một stay có thể kéo dài qua ranh giới thời gian.

Ví dụ:

```text
20:50 ───────── 21:30
         21:00 bắt đầu HOME window
```

Trong trường hợp này chỉ có **30 phút từ 21:00–21:30** là HOME evidence.

Nếu chỉ nhìn giờ đến `20:50`, ta sẽ bỏ mất phần evidence đó.
Nếu chỉ nhìn giờ rời đi rồi gán cả stay là HOME, ta lại tính dư 10 phút.

Vì vậy notebook tính **phần thời gian thực sự overlap với HOME/OFFICE window**.

### Mỗi recurring location được chấm bằng những evidence nào?

Với mỗi location, ta đo:

* tổng thời gian nằm trong HOME hoặc OFFICE window;
* tỷ lệ thời gian đó so với toàn bộ semantic dwell của user;
* số ngày khác nhau có ít nhất **10 phút overlap**;
* khoảng cách giữa location đứng đầu và đứng thứ hai (`margin`).

`share` = evidence của location / tổng semantic dwell của user.

Tổng này bao gồm **mọi semantic locations**, không chỉ recurring candidates, để tránh làm `share` bị phóng đại.

Nhờ vậy, nếu user có nhiều dwell rải rác ở các location nhỏ khác, phần evidence đó vẫn được tính vào tổng và không bị “biến mất”.

### HOME và OFFICE có bắt buộc là hai nơi khác nhau không?

Không.

Nếu cùng một location vừa đứng đầu về night evidence, vừa đứng đầu về weekday daytime evidence, notebook vẫn giữ kết quả đó thay vì cố tạo ra một location thứ hai.

Trường hợp này được xem là **ambiguous but valid evidence**, và sẽ được report để audit.



In [ ]:
HOME_NIGHT_START_HOUR = 21
HOME_NIGHT_END_HOUR = 6
OFFICE_START_HOUR = 9
OFFICE_END_HOUR = 17
OFFICE_WEEKDAYS = {0, 1, 2, 3, 4}

MIN_RELEVANT_DATE_OVERLAP_S = 10 * 60
MIN_RELEVANT_DATES = 2

def interval_overlap_s(start, end, window_start, window_end):
    overlap_start = max(start, window_start)
    overlap_end = min(end, window_end)
    if overlap_end <= overlap_start:
        return 0.0
    return float((overlap_end - overlap_start).total_seconds())

def stay_window_contributions(row):
    start = row.arrival_time_local
    end = row.departure_time_local

    night_rows = []
    office_rows = []

    # A stay shortly after midnight can overlap the night window that started
    # on the previous local date, so include one padded date before arrival.
    day = start.normalize() - pd.Timedelta(days=1)
    last_day = end.normalize()

    while day <= last_day:
        night_start = day + pd.Timedelta(hours=HOME_NIGHT_START_HOUR)
        night_end = day + pd.Timedelta(days=1, hours=HOME_NIGHT_END_HOUR)
        night_s = interval_overlap_s(start, end, night_start, night_end)
        if night_s > 0:
            night_rows.append(
                {
                    "user_id": row.user_id,
                    "location_id": int(row.location_id),
                    "behavior_date": night_start.date(),
                    "overlap_s": night_s,
                }
            )

        if day.weekday() in OFFICE_WEEKDAYS:
            office_start = day + pd.Timedelta(hours=OFFICE_START_HOUR)
            office_end = day + pd.Timedelta(hours=OFFICE_END_HOUR)
            office_s = interval_overlap_s(start, end, office_start, office_end)
            if office_s > 0:
                office_rows.append(
                    {
                        "user_id": row.user_id,
                        "location_id": int(row.location_id),
                        "behavior_date": office_start.date(),
                        "overlap_s": office_s,
                    }
                )

        day += pd.Timedelta(days=1)

    return night_rows, office_rows

night_rows = []
office_rows = []

for row in semantic_stays.itertuples(index=False):
    nr, orows = stay_window_contributions(row)
    night_rows.extend(nr)
    office_rows.extend(orows)

night_contrib = pd.DataFrame(
    night_rows,
    columns=["user_id", "location_id", "behavior_date", "overlap_s"],
)
office_contrib = pd.DataFrame(
    office_rows,
    columns=["user_id", "location_id", "behavior_date", "overlap_s"],
)

def aggregate_relevant_window(contrib, prefix):
    if contrib.empty:
        return pd.DataFrame(
            columns=[
                "user_id",
                "location_id",
                f"{prefix}_dwell_s",
                f"{prefix}_dates",
            ]
        )

    per_date = (
        contrib.groupby(["user_id", "location_id", "behavior_date"], as_index=False)
        ["overlap_s"]
        .sum()
    )

    dwell = (
        per_date.groupby(["user_id", "location_id"], as_index=False)["overlap_s"]
        .sum()
        .rename(columns={"overlap_s": f"{prefix}_dwell_s"})
    )

    supported_dates = (
        per_date.loc[per_date["overlap_s"] >= MIN_RELEVANT_DATE_OVERLAP_S]
        .groupby(["user_id", "location_id"], as_index=False)["behavior_date"]
        .nunique()
        .rename(columns={"behavior_date": f"{prefix}_dates"})
    )

    return dwell.merge(
        supported_dates,
        on=["user_id", "location_id"],
        how="left",
    ).fillna({f"{prefix}_dates": 0})

night_features = aggregate_relevant_window(night_contrib, "night")
office_features = aggregate_relevant_window(office_contrib, "office")

semantic_features = (
    semantic_locations.merge(
        night_features,
        on=["user_id", "location_id"],
        how="left",
    )
    .merge(
        office_features,
        on=["user_id", "location_id"],
        how="left",
    )
)

for col in ["night_dwell_s", "night_dates", "office_dwell_s", "office_dates"]:
    semantic_features[col] = semantic_features[col].fillna(0)

semantic_features["night_dates"] = semantic_features["night_dates"].astype(int)
semantic_features["office_dates"] = semantic_features["office_dates"].astype(int)

user_night_total = semantic_features.groupby("user_id")["night_dwell_s"].transform("sum")
user_office_total = semantic_features.groupby("user_id")["office_dwell_s"].transform("sum")

semantic_features["night_dwell_share"] = np.where(
    user_night_total > 0,
    semantic_features["night_dwell_s"] / user_night_total,
    0.0,
)
semantic_features["office_dwell_share"] = np.where(
    user_office_total > 0,
    semantic_features["office_dwell_s"] / user_office_total,
    0.0,
)

semantic_features["night_dwell_h"] = semantic_features["night_dwell_s"] / 3600.0
semantic_features["office_dwell_h"] = semantic_features["office_dwell_s"] / 3600.0

def rank_semantic_candidates(
    features,
    *,
    share_col,
    dates_col,
    dwell_col,
    label,
):
    eligible = features[
        (features["stay_count"] >= 2)
        & (features[dates_col] >= MIN_RELEVANT_DATES)
        & (features[dwell_col] > 0)
    ].copy()

    if eligible.empty:
        return eligible

    eligible = eligible.sort_values(
        ["user_id", share_col, dates_col, dwell_col, "stay_count"],
        ascending=[True, False, False, False, False],
        kind="stable",
    )
    eligible[f"{label}_rank"] = eligible.groupby("user_id").cumcount() + 1
    return eligible

home_ranked = rank_semantic_candidates(
    semantic_features,
    share_col="night_dwell_share",
    dates_col="night_dates",
    dwell_col="night_dwell_s",
    label="home",
)
office_ranked = rank_semantic_candidates(
    semantic_features,
    share_col="office_dwell_share",
    dates_col="office_dates",
    dwell_col="office_dwell_s",
    label="office",
)

def top_with_margin(ranked, *, label, share_col):
    if ranked.empty:
        return pd.DataFrame()

    top1 = ranked[ranked[f"{label}_rank"] == 1].copy()
    second = (
        ranked[ranked[f"{label}_rank"] == 2][["user_id", share_col]]
        .rename(columns={share_col: f"{label}_second_share"})
    )
    top1 = top1.merge(second, on="user_id", how="left")
    top1[f"{label}_second_share"] = top1[f"{label}_second_share"].fillna(0.0)
    top1[f"{label}_share_margin"] = (
        top1[share_col] - top1[f"{label}_second_share"]
    )
    return top1

home_top = top_with_margin(
    home_ranked,
    label="home",
    share_col="night_dwell_share",
)
office_top = top_with_margin(
    office_ranked,
    label="office",
    share_col="office_dwell_share",
)

candidate_users = set(stays_semantic["user_id"].unique())
home_users = set(home_top["user_id"]) if not home_top.empty else set()
office_users = set(office_top["user_id"]) if not office_top.empty else set()
both_users = home_users & office_users

print("Timezone-v2 candidate cohort users:", len(candidate_users))
print("Users with recurring semantic location:", semantic_locations.loc[
    semantic_locations["stay_count"] >= 2, "user_id"
].nunique())
print("Users with supported HOME candidate:", len(home_users))
print("Users with supported OFFICE candidate:", len(office_users))
print("Users with both candidates:", len(both_users))
print("Users abstaining from HOME:", len(candidate_users - home_users))
print("Users abstaining from OFFICE:", len(candidate_users - office_users))

if both_users:
    paired = (
        home_top[home_top["user_id"].isin(both_users)][
            ["user_id", "location_id"]
        ]
        .rename(columns={"location_id": "home_location_id"})
        .merge(
            office_top[office_top["user_id"].isin(both_users)][
                ["user_id", "location_id"]
            ].rename(columns={"location_id": "office_location_id"}),
            on="user_id",
        )
    )
    paired["same_location_candidate"] = (
        paired["home_location_id"] == paired["office_location_id"]
    )
    print(
        "Both-candidate users with same leading location:",
        int(paired["same_location_candidate"].sum()),
        "/",
        len(paired),
    )

def candidate_distribution(top, cols):
    if top.empty:
        return pd.DataFrame()
    return top[cols].describe(
        percentiles=[.1, .25, .5, .75, .9, .95]
    )

print("\nHOME top-candidate evidence:")
display(
    candidate_distribution(
        home_top,
        [
            "night_dwell_share",
            "home_share_margin",
            "night_dates",
            "night_dwell_h",
            "stay_count",
        ],
    )
)

print("\nOFFICE top-candidate evidence:")
display(
    candidate_distribution(
        office_top,
        [
            "office_dwell_share",
            "office_share_margin",
            "office_dates",
            "office_dwell_h",
            "stay_count",
        ],
    )
)

print("\nSample HOME candidates (no coordinates displayed):")
display(
    home_top[
        [
            "user_id",
            "location_id",
            "stay_count",
            "active_local_dates",
            "night_dates",
            "night_dwell_h",
            "night_dwell_share",
            "home_share_margin",
        ]
    ].head(20)
)

print("\nSample OFFICE candidates (no coordinates displayed):")
display(
    office_top[
        [
            "user_id",
            "location_id",
            "stay_count",
            "active_local_dates",
            "office_dates",
            "office_dwell_h",
            "office_dwell_share",
            "office_share_margin",
        ]
    ].head(20)
)


### 6.0 Diễn giải scoring sau timezone migration

Không reuse các số `97 cohort users / 73 recurring users / 47 HOME / 40 OFFICE candidates` từ geography/timezone cohort cũ.

Cell phía trên tính lại trên candidate mới:

```text
coordinate-based timezone lookup
→ Asia/Shanghai-focused users
→ travel stays giữ timezone thật
→ recurring locations
→ HOME/OFFICE behavioral evidence
```

Các số mới sau rerun mới là evidence cho migration.


### 6.1 Kiểm tra độ ổn định của scoring

GeoLife không có ground truth HOME/OFFICE, nên không maximize accuracy.

Sau khi đổi timezone semantics, phải **rerun** hai sensitivity audit cũ thay vì reuse phần trăm cũ.

HOME windows: `20–06 / 21–06 / 22–06`.  
OFFICE windows: `08–17 / 09–17 / 09–18`.

Emission grids vẫn thử các mức dates/share/margin như trước.

Các v1 middle settings được giữ làm controls. Nếu output mới khác đáng kể, phải review lại trước khi production migration.


In [ ]:
def build_semantic_features_for_windows(
    *,
    home_start_hour,
    home_end_hour,
    office_start_hour,
    office_end_hour,
):
    night_rows = []
    office_rows = []

    for row in semantic_stays.itertuples(index=False):
        start = row.arrival_time_local
        end = row.departure_time_local

        day = start.normalize() - pd.Timedelta(days=1)
        last_day = end.normalize()

        while day <= last_day:
            night_start = day + pd.Timedelta(hours=home_start_hour)
            night_end = day + pd.Timedelta(days=1, hours=home_end_hour)
            night_s = interval_overlap_s(start, end, night_start, night_end)
            if night_s > 0:
                night_rows.append(
                    {
                        "user_id": row.user_id,
                        "location_id": int(row.location_id),
                        "behavior_date": night_start.date(),
                        "overlap_s": night_s,
                    }
                )

            if day.weekday() in OFFICE_WEEKDAYS:
                office_start = day + pd.Timedelta(hours=office_start_hour)
                office_end = day + pd.Timedelta(hours=office_end_hour)
                office_s = interval_overlap_s(start, end, office_start, office_end)
                if office_s > 0:
                    office_rows.append(
                        {
                            "user_id": row.user_id,
                            "location_id": int(row.location_id),
                            "behavior_date": office_start.date(),
                            "overlap_s": office_s,
                        }
                    )

            day += pd.Timedelta(days=1)

    night_contrib_local = pd.DataFrame(
        night_rows,
        columns=["user_id", "location_id", "behavior_date", "overlap_s"],
    )
    office_contrib_local = pd.DataFrame(
        office_rows,
        columns=["user_id", "location_id", "behavior_date", "overlap_s"],
    )

    night_features_local = aggregate_relevant_window(
        night_contrib_local,
        "night",
    )
    office_features_local = aggregate_relevant_window(
        office_contrib_local,
        "office",
    )

    features = (
        semantic_locations.merge(
            night_features_local,
            on=["user_id", "location_id"],
            how="left",
        )
        .merge(
            office_features_local,
            on=["user_id", "location_id"],
            how="left",
        )
    )

    for col in ["night_dwell_s", "night_dates", "office_dwell_s", "office_dates"]:
        features[col] = features[col].fillna(0)

    features["night_dates"] = features["night_dates"].astype(int)
    features["office_dates"] = features["office_dates"].astype(int)

    user_night_total = features.groupby("user_id")["night_dwell_s"].transform("sum")
    user_office_total = features.groupby("user_id")["office_dwell_s"].transform("sum")

    features["night_dwell_share"] = np.where(
        user_night_total > 0,
        features["night_dwell_s"] / user_night_total,
        0.0,
    )
    features["office_dwell_share"] = np.where(
        user_office_total > 0,
        features["office_dwell_s"] / user_office_total,
        0.0,
    )

    features["night_dwell_h"] = features["night_dwell_s"] / 3600.0
    features["office_dwell_h"] = features["office_dwell_s"] / 3600.0
    return features


def top_candidates_for(
    features,
    *,
    label,
    min_dates,
):
    if label == "home":
        share_col = "night_dwell_share"
        dates_col = "night_dates"
        dwell_col = "night_dwell_s"
    elif label == "office":
        share_col = "office_dwell_share"
        dates_col = "office_dates"
        dwell_col = "office_dwell_s"
    else:
        raise ValueError(label)

    ranked = features[
        (features["stay_count"] >= 2)
        & (features[dates_col] >= min_dates)
        & (features[dwell_col] > 0)
    ].copy()

    if ranked.empty:
        return ranked

    ranked = ranked.sort_values(
        ["user_id", share_col, dates_col, dwell_col, "stay_count"],
        ascending=[True, False, False, False, False],
        kind="stable",
    )
    ranked[f"{label}_rank"] = ranked.groupby("user_id").cumcount() + 1
    return top_with_margin(
        ranked,
        label=label,
        share_col=share_col,
    )


def window_stability_row(
    top,
    *,
    baseline_top,
    label,
    variant,
    share_col,
    margin_col,
    dates_col,
):
    if top.empty:
        return {
            "variant": variant,
            "supported_users": 0,
            "shared_with_baseline": 0,
            "same_top_location_rate": np.nan,
            "median_share": np.nan,
            "median_margin": np.nan,
            "median_dates": np.nan,
        }

    current = top[["user_id", "location_id"]].rename(
        columns={"location_id": "current_location_id"}
    )
    base = baseline_top[["user_id", "location_id"]].rename(
        columns={"location_id": "baseline_location_id"}
    )
    shared = current.merge(base, on="user_id", how="inner")

    same_rate = (
        float(
            (shared["current_location_id"] == shared["baseline_location_id"]).mean()
        )
        if len(shared)
        else np.nan
    )

    return {
        "variant": variant,
        "supported_users": len(top),
        "shared_with_baseline": len(shared),
        "same_top_location_rate": same_rate,
        "median_share": float(top[share_col].median()),
        "median_margin": float(top[margin_col].median()),
        "median_dates": float(top[dates_col].median()),
    }


HOME_WINDOW_VARIANTS = [
    ("20-06", 20, 6),
    ("21-06", 21, 6),
    ("22-06", 22, 6),
]
OFFICE_WINDOW_VARIANTS = [
    ("08-17", 8, 17),
    ("09-17", 9, 17),
    ("09-18", 9, 18),
]

home_window_rows = []
for name, start_hour, end_hour in HOME_WINDOW_VARIANTS:
    features_variant = build_semantic_features_for_windows(
        home_start_hour=start_hour,
        home_end_hour=end_hour,
        office_start_hour=OFFICE_START_HOUR,
        office_end_hour=OFFICE_END_HOUR,
    )
    top_variant = top_candidates_for(
        features_variant,
        label="home",
        min_dates=2,
    )
    home_window_rows.append(
        window_stability_row(
            top_variant,
            baseline_top=home_top,
            label="home",
            variant=name,
            share_col="night_dwell_share",
            margin_col="home_share_margin",
            dates_col="night_dates",
        )
    )

office_window_rows = []
for name, start_hour, end_hour in OFFICE_WINDOW_VARIANTS:
    features_variant = build_semantic_features_for_windows(
        home_start_hour=HOME_NIGHT_START_HOUR,
        home_end_hour=HOME_NIGHT_END_HOUR,
        office_start_hour=start_hour,
        office_end_hour=end_hour,
    )
    top_variant = top_candidates_for(
        features_variant,
        label="office",
        min_dates=2,
    )
    office_window_rows.append(
        window_stability_row(
            top_variant,
            baseline_top=office_top,
            label="office",
            variant=name,
            share_col="office_dwell_share",
            margin_col="office_share_margin",
            dates_col="office_dates",
        )
    )

print("HOME window stability:")
display(pd.DataFrame(home_window_rows))

print("OFFICE window stability:")
display(pd.DataFrame(office_window_rows))


recurring_user_count = int(
    semantic_locations.loc[
        semantic_locations["stay_count"] >= 2,
        "user_id",
    ].nunique()
)

def emission_grid(
    features,
    *,
    label,
    min_dates_values,
    min_share_values,
    min_margin_values,
):
    if label == "home":
        share_col = "night_dwell_share"
        margin_col = "home_share_margin"
    elif label == "office":
        share_col = "office_dwell_share"
        margin_col = "office_share_margin"
    else:
        raise ValueError(label)

    rows = []
    for min_dates in min_dates_values:
        top = top_candidates_for(
            features,
            label=label,
            min_dates=min_dates,
        )

        for min_share in min_share_values:
            for min_margin in min_margin_values:
                emitted = top[
                    (top[share_col] >= min_share)
                    & (top[margin_col] >= min_margin)
                ]
                rows.append(
                    {
                        "min_dates": min_dates,
                        "min_share": min_share,
                        "min_margin": min_margin,
                        "emitted_users": len(emitted),
                        "cohort_coverage": len(emitted) / len(candidate_users),
                        "recurring_user_coverage": (len(emitted) / recurring_user_count if recurring_user_count else np.nan),
                    }
                )
    return pd.DataFrame(rows)


home_emission_sensitivity = emission_grid(
    semantic_features,
    label="home",
    min_dates_values=[2, 3, 5],
    min_share_values=[0.4, 0.5, 0.6],
    min_margin_values=[0.1, 0.2, 0.3],
)

office_emission_sensitivity = emission_grid(
    semantic_features,
    label="office",
    min_dates_values=[2, 3, 5],
    min_share_values=[0.2, 0.3, 0.4],
    min_margin_values=[0.05, 0.10, 0.20],
)

print("HOME emission sensitivity:")
display(home_emission_sensitivity)

print("OFFICE emission sensitivity:")
display(office_emission_sensitivity)

# -------------------------------------------------------------------
# Candidate v2 emissions under the old v1 scoring gates (fixed control)
# -------------------------------------------------------------------

def apply_candidate_gate(
    features,
    *,
    label,
    min_dates,
    min_share,
    min_margin,
):
    top = top_candidates_for(
        features,
        label=label,
        min_dates=min_dates,
    )

    if top.empty:
        return top

    if label == "home":
        share_col = "night_dwell_share"
        margin_col = "home_share_margin"
    elif label == "office":
        share_col = "office_dwell_share"
        margin_col = "office_share_margin"
    else:
        raise ValueError(label)

    emitted = top[
        (top[share_col] >= min_share)
        & (top[margin_col] >= min_margin)
    ].copy()

    emitted["label"] = label.upper()
    emitted["relevant_dwell_share"] = emitted[share_col]
    emitted["share_margin"] = emitted[margin_col]

    date_col = "night_dates" if label == "home" else "office_dates"
    emitted["relevant_dates"] = emitted[date_col].astype(int)

    support_factor = np.minimum(
        emitted["relevant_dates"].to_numpy(dtype=float) / 5.0,
        1.0,
    )

    emitted["evidence_strength"] = (
        emitted["relevant_dwell_share"].to_numpy(dtype=float)
        + emitted["share_margin"].to_numpy(dtype=float)
        + support_factor
    ) / 3.0

    return emitted


candidate_home_emitted = apply_candidate_gate(
    semantic_features,
    label="home",
    min_dates=3,
    min_share=0.50,
    min_margin=0.20,
)

candidate_office_emitted = apply_candidate_gate(
    semantic_features,
    label="office",
    min_dates=3,
    min_share=0.30,
    min_margin=0.10,
)

candidate_emission_counts = pd.Series(
    {
        "HOME": len(candidate_home_emitted),
        "OFFICE": len(candidate_office_emitted),
    },
    name="candidate_v2_emitted_users",
)

print("\nTimezone-v2 candidate emissions under fixed v1 scoring gates:")
display(candidate_emission_counts.to_frame())


### Cách đọc share, margin và support

`relevant_dwell_share` trả lời: location này chiếm bao nhiêu phần evidence window của user.

`share_margin` trả lời: top location có tách khỏi runner-up rõ không.

`relevant_dates` trả lời: evidence có repeat qua nhiều ngày hay chỉ đến từ một episode dài.

Ba signals bổ sung cho nhau:

```text
share cao nhưng margin thấp → hai locations cạnh tranh gần ngang nhau
margin cao nhưng 1–2 dates → evidence còn mỏng
nhiều dates nhưng share thấp → behavior phân tán
```

Vì vậy final emission dùng **gate**, còn aggregate confidence chỉ là evidence-strength summary.

### 6.2 Scoring gates đang là fixed controls, chưa refreeze

Trong migration này ta **không đổi đồng thời** timezone rule và scoring rule.

Tạm giữ HOME `21–06 / 3 dates / 0.50 share / 0.20 margin` và OFFICE `Mon–Fri 09–17 / 3 dates / 0.30 share / 0.10 margin`.

Cell sensitivity phải được rerun để xem top location còn stable không và candidate emissions khác production v1 `27 HOME / 16 OFFICE` bao nhiêu.

`evidence_strength` vẫn chỉ là heuristic summary, không phải calibrated probability.


### 6.3 So sánh candidate notebook với production v1

Production code hiện tại vẫn implement CP2 v1 geography rule:

```text
Beijing reference point
+ 100 km
+ 80/80 share
```

Notebook candidate đã chuyển sang coordinate → IANA timezone + Asia/Shanghai concentration + per-stay local time.

Vì vậy **không được assert candidate mới phải bằng `27 HOME / 16 OFFICE`**.

Cell dưới chỉ chạy production v1 như một **historical reference** rồi đặt cạnh candidate counts. Migration chỉ được productionize sau khi sensitivity, output delta và tests được review.


In [ ]:
# Current production v1 reference.
# This is intentionally NOT asserted equal to the timezone-v2 notebook candidate.

production_v1_config = HomeOfficeConfig()
production_v1_labels = infer_home_office(
    stays,
    config=production_v1_config,
)

production_v1_counts = (
    production_v1_labels["label"]
    .value_counts()
    .reindex(["HOME", "OFFICE"], fill_value=0)
)

migration_comparison = pd.DataFrame(
    {
        "production_v1": [
            int(production_v1_counts["HOME"]),
            int(production_v1_counts["OFFICE"]),
        ],
        "timezone_v2_candidate": [
            int(candidate_emission_counts["HOME"]),
            int(candidate_emission_counts["OFFICE"]),
        ],
    },
    index=["HOME", "OFFICE"],
)

migration_comparison["delta_candidate_minus_v1"] = (
    migration_comparison["timezone_v2_candidate"]
    - migration_comparison["production_v1"]
)

display(migration_comparison)

print("Production v1 rows:", len(production_v1_labels))
print(
    "Production v1 unique users:",
    production_v1_labels["user_id"].nunique(),
)
print(
    "Candidate v2 unique emitted users:",
    len(
        set(candidate_home_emitted.get("user_id", pd.Series(dtype=str)))
        | set(candidate_office_emitted.get("user_id", pd.Series(dtype=str)))
    ),
)

print(
    "\nSTATUS: comparison only. "
    "A difference is expected to trigger review, not an assertion failure."
)


### 6.4 DBSCAN end-to-end production benchmark — tạm khóa trong migration

Production `HomeOfficeConfig` hiện chưa có contract coordinate → IANA timezone per stay.

Vì vậy dùng production `build_semantic_locations()` để benchmark lúc này sẽ quay lại timezone/geography v1, không còn cùng input với notebook candidate.

Migration notebook **không chạy production DBSCAN end-to-end benchmark** ở cell tiếp theo. Spatial DBSCAN-vs-complete-link benchmark ở Section 4 vẫn chạy trên candidate cohort mới.


In [ ]:
print(
    "SKIPPED BY DESIGN: production HomeOfficeConfig still implements "
    "the CP2 v1 Beijing-radius timezone/geography contract."
)
print(
    "Run the end-to-end DBSCAN benchmark again only after "
    "coordinate->IANA timezone assignment is implemented in src/ "
    "and covered by tests."
)


### Handoff cho end-to-end benchmark

Hiện tại:

```text
notebook candidate
→ timezone-v2 semantics

production code
→ timezone/geography-v1 semantics
```

nên benchmark end-to-end qua production code sẽ là apples-to-oranges.

Khi production được migrate, benchmark phải giữ cùng timezone assignment, user concentration gate, travel-stay handling, clustering alternatives, time windows và emission gates.


## 7. Kết luận timezone-v2 candidate

### Candidate pipeline mới

```text
5,821 frozen CP1 stays
   ↓
coordinate → timezone polygon
IANA timezone_id per stay
   ↓
Asia/Shanghai stay-share + dwell-share
   ↓
retain resolved stays of eligible users
travel stays keep their own timezone
   ↓
per-stay UTC → local wall time
   ↓
DBSCAN vs complete-link
   ↓
complete-link 200 m fixed migration control
   ↓
HOME 21–06 / OFFICE weekday 09–17
   ↓
fixed v1 emission gates for comparison
candidate HOME / OFFICE counts
   ↓
compare against production v1
```

### Điều đã sửa

- bỏ `100 km quanh một Beijing point` như timezone proxy;
- timezone lookup từ `(lat, lon)` của từng stay;
- travel stay giữ timezone thật;
- local time tính theo timezone của chính stay;
- `Asia/Shanghai-focused` và `Beijing geographic membership` được tách thành hai khái niệm;
- các output hard-code từ cohort cũ không còn được reuse như evidence của candidate mới.

### Điều chưa đổi trong production

`src/geolife/model/home_office.py` vẫn là CP2 v1 để giữ reproducibility.

Notebook candidate phải được **Run All** trên full 5,821 stays, rồi review timezone coverage, cohort sensitivity, clustering, scoring stability và emission delta.

Nếu requirement thật sự là “chỉ infer users thuộc Beijing”, bước tiếp theo là **Beijing geography audit riêng**, ưu tiên administrative polygon. Timezone polygon trả lời “đồng hồ địa phương nào?”, không trả lời “có nằm trong Beijing municipality không?”.
